# GoGame-Detection: запуск в Google Colab

Этот ноутбук по шагам:
- устанавливает `uv`;
- клонирует/обновляет репозиторий `GoGame-Detection` (ветка `convert-videofile`);
- настраивает версию Python 3.9 через `.python-version` и устанавливает её через `uv`;
- создаёт виртуальное окружение с `uv venv` и ставит зависимости;
- запускает конвертацию видеофайла в SGF.

Вывод команд отображается по мере выполнения, так как используются прямые вызовы через `!`, а не `%%bash`.

In [ ]:
# Если задать путь здесь, диалог загрузки файла открываться не будет.
VIDEO_PATH = "/content/drive/MyDrive/Go/my_game.mp4"

In [ ]:
import os

VIDEO_PATH = globals().get("VIDEO_PATH", "")
print("Текущий VIDEO_PATH:", repr(VIDEO_PATH))

# Если путь указывает на Google Drive и диск ещё не смонтирован — монтируем его
if VIDEO_PATH.startswith("/content/drive") and not os.path.ismount("/content/drive"):
    from google.colab import drive
    print("VIDEO_PATH указывает на Google Drive, монтируем /content/drive ...")
    drive.mount("/content/drive")

In [ ]:
# Ячейка 2 — загрузка видеофайла через интерфейс Colab (если VIDEO_PATH не задан вручную)
from google.colab import files
import os

if globals().get("VIDEO_PATH"):
    print("VIDEO_PATH уже задан вручную:", VIDEO_PATH)
else:
    print("VIDEO_PATH не задан, откроется диалог загрузки файла...")
    uploaded = files.upload()

    if uploaded:
        # Берём первое загруженное имя файла
        filename = next(iter(uploaded.keys()))
        VIDEO_PATH = os.path.join("/content", filename)
        print(f"VIDEO_PATH установлен в: {VIDEO_PATH}")
    else:
        VIDEO_PATH = ""
        print("Файл не загружен. Установите VIDEO_PATH вручную в одной из следующих ячеек.")

In [ ]:
# Ячейка 1 — установка uv (один раз на среду)
import os, sys

os.chdir("/content")

if os.system("which uv > /dev/null 2>&1") != 0:
    print("Установка uv...")
    os.system("curl -LsSf https://astral.sh/uv/install.sh | sh")
    os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ.get("PATH", "")
else:
    os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ.get("PATH", "")

!uv --version

In [ ]:
# Ячейка 2 — клонирование / обновление репозитория
import os

os.chdir("/content")

if os.path.isdir("GoGame-Detection"):
    print("Обновляю репозиторий...")
    !cd /content/GoGame-Detection && git fetch origin
    !cd /content/GoGame-Detection && git checkout convert-videofile 2>/dev/null || true
    !cd /content/GoGame-Detection && git pull origin convert-videofile 2>/dev/null || true
else:
    print("Клонирую репозиторий...")
    !git clone https://github.com/Wzhoooh/GoGame-Detection.git
    !cd /content/GoGame-Detection && git checkout convert-videofile 2>/dev/null || true

print("Содержимое /content/GoGame-Detection:")
!ls /content/GoGame-Detection

In [ ]:
# Ячейка 3 — установка Python 3.9 через uv
import os

os.chdir("/content/GoGame-Detection")

print("Устанавливаю Python 3.9 через uv...")
!uv python install 3.9

In [ ]:
# Ячейка 4 — создание venv и установка зависимостей
import os

os.chdir("/content/GoGame-Detection")

print("Создаю venv...")
!uv venv .venv

PYTHON_BIN = "/content/GoGame-Detection/.venv/bin/python"

print("Python в venv:")
!{PYTHON_BIN} --version

print("Устанавливаю зависимости...")
!uv pip install -r requirements.txt --python {PYTHON_BIN}

In [ ]:
# Ячейка 6 — запуск конвертера видео → SGF
import os

# Если в предыдущей ячейке файл не загружен, можно явно задать путь здесь:
# VIDEO_PATH = "/content/drive/MyDrive/Го/video_2026-03-15_21-24-20.mp4"

DEVICE = ""  # можно '', 'cuda', 'cpu'

os.chdir("/content/GoGame-Detection")

PYTHON_BIN = "/content/GoGame-Detection/.venv/bin/python"

if not globals().get("VIDEO_PATH"):
    raise RuntimeError("VIDEO_PATH не задан. Загрузите файл в предыдущей ячейке или установите VIDEO_PATH вручную.")

print("Используем VIDEO_PATH:", VIDEO_PATH)
print("----запуск----")
if DEVICE:
    !{PYTHON_BIN} main.py "{VIDEO_PATH}" --device {DEVICE}
else:
    !{PYTHON_BIN} main.py "{VIDEO_PATH}"